In [ ]:
# ============================================================
# Setup
# ============================================================

import subprocess
import sys

REPOSITORY_URL = "https://github.com/MRamazan/krea2-character-lora.git"
PIPELINE_REVISION = "main"
WORKSPACE = "/content/krea2_character_lora"

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--upgrade",
        f"git+{REPOSITORY_URL}@{PIPELINE_REVISION}",
    ]
)

from krea2_character_lora import CharacterLoraPipeline

pipeline = CharacterLoraPipeline(
    workspace=WORKSPACE,
    repository_revision=PIPELINE_REVISION,
)

setup_report = pipeline.setup(
    verify_environment=True,
    prepare_training_assets=True,
    prepare_inference_assets=False,
)

setup_report.display()

In [ ]:
# ============================================================
# Dataset
# ============================================================

from google.colab import files

from krea2_character_lora import DatasetConfig

TRIGGER_WORD = "mycharacter"
CAPTION_TRIGGER_POLICY = "require"
AUTO_PREFIX_MISSING_TRIGGER = False
MINIMUM_PAIR_COUNT = 4
EXPECTED_PAIR_COUNT = None
FAIL_ON_EXACT_DUPLICATES = True
NEAR_DUPLICATE_HAMMING_THRESHOLD = 8
ACCEPTED_IMAGE_EXTENSIONS = (".png", ".jpg", ".jpeg")
GALLERY_COLUMNS = 4
GALLERY_THUMBNAIL_SIZE = 320
GALLERY_PAGE_SIZE = 24

uploaded = files.upload()
zip_files = [name for name in uploaded if name.lower().endswith(".zip")]

if len(zip_files) != 1:
    raise RuntimeError("Upload exactly one dataset ZIP file.")

DATASET_ZIP = f"/content/{zip_files[0]}"

dataset_config = DatasetConfig(
    trigger_word=TRIGGER_WORD,
    caption_trigger_policy=CAPTION_TRIGGER_POLICY,
    auto_prefix_missing_trigger=AUTO_PREFIX_MISSING_TRIGGER,
    minimum_pair_count=MINIMUM_PAIR_COUNT,
    expected_pair_count=EXPECTED_PAIR_COUNT,
    fail_on_exact_duplicates=FAIL_ON_EXACT_DUPLICATES,
    near_duplicate_hamming_threshold=NEAR_DUPLICATE_HAMMING_THRESHOLD,
    accepted_image_extensions=ACCEPTED_IMAGE_EXTENSIONS,
    gallery_columns=GALLERY_COLUMNS,
    gallery_thumbnail_size=GALLERY_THUMBNAIL_SIZE,
    gallery_page_size=GALLERY_PAGE_SIZE,
)

dataset = pipeline.prepare_dataset(
    zip_path=DATASET_ZIP,
    config=dataset_config,
)

dataset.display_summary()
dataset.show_gallery(
    columns=GALLERY_COLUMNS,
    thumbnail_size=GALLERY_THUMBNAIL_SIZE,
    page_size=GALLERY_PAGE_SIZE,
    show_filename=True,
    show_dimensions=True,
    show_caption=True,
    highlight_trigger=True,
)
dataset.show_caption_audit()
dataset.show_issues()

In [ ]:
# ============================================================
# Training
# ============================================================

import os
from pathlib import Path

from krea2_character_lora import TrainingConfig

PROJECT_NAME = "krea2_character_lora"
RUN_NAME = "character_v1"
TRAINING_STEPS = 2000
LEARNING_RATE = 0.0001
WEIGHT_DECAY = 0.0001
BATCH_SIZE = 1
GRADIENT_ACCUMULATION = 1
TRAINING_RESOLUTIONS = (768, 1024)
LORA_RANK = 32
LORA_ALPHA = 32
SAVE_EVERY = 200
MAX_CHECKPOINTS_TO_KEEP = 10
DATASET_REPEATS = 1
CAPTION_DROPOUT_RATE = 0.0
TOKEN_DROPOUT_RATE = 0.0
SHUFFLE_TOKENS = False
KEEP_TOKENS = 1
FLIP_X = False
TRAINING_DTYPE = "bf16"
GENERATE_TRAINING_SAMPLES = False
TRAINING_SAMPLE_EVERY = 200
RUN_SMOKE_TEST = True
SMOKE_TEST_STEPS = 3
RUN_PRODUCTION_TRAINING = True
RESUME_MODE = "auto"

training_config = TrainingConfig(
    project_name=PROJECT_NAME,
    run_name=RUN_NAME,
    training_steps=TRAINING_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    batch_size=BATCH_SIZE,
    gradient_accumulation=GRADIENT_ACCUMULATION,
    resolutions=TRAINING_RESOLUTIONS,
    lora_rank=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    save_every=SAVE_EVERY,
    max_checkpoints_to_keep=MAX_CHECKPOINTS_TO_KEEP,
    dataset_repeats=DATASET_REPEATS,
    caption_dropout_rate=CAPTION_DROPOUT_RATE,
    token_dropout_rate=TOKEN_DROPOUT_RATE,
    shuffle_tokens=SHUFFLE_TOKENS,
    keep_tokens=KEEP_TOKENS,
    flip_x=FLIP_X,
    training_dtype=TRAINING_DTYPE,
    generate_training_samples=GENERATE_TRAINING_SAMPLES,
    training_sample_every=TRAINING_SAMPLE_EVERY,
)

pipeline.preview_training(
    dataset=dataset,
    config=training_config,
)

AI_TOOLKIT_DIR = Path(pipeline.paths.ai_toolkit)

assert AI_TOOLKIT_DIR.exists(), AI_TOOLKIT_DIR
assert (AI_TOOLKIT_DIR / "extensions_built_in").exists()

current_pythonpath = os.environ.get("PYTHONPATH", "")
os.environ["PYTHONPATH"] = (
    f"{AI_TOOLKIT_DIR}:{current_pythonpath}" if current_pythonpath else str(AI_TOOLKIT_DIR)
)


training_run = pipeline.train(
    dataset=dataset,
    config=training_config,
    run_smoke_test=RUN_SMOKE_TEST,
    smoke_test_steps=SMOKE_TEST_STEPS,
    run_production=RUN_PRODUCTION_TRAINING,
    resume=RESUME_MODE,
)

training_run.display_summary()
training_run.show_checkpoints()

In [ ]:
# ============================================================
# Evaluation and export
# ============================================================

from krea2_character_lora import EvaluationConfig

PROMPTS = [
    f"{training_run.trigger_word} is a woman in a fully clothed professional studio portrait, "
    "wearing a tailored blazer, soft key lighting, sharp facial detail",
    f"{training_run.trigger_word} is a woman standing outdoors in natural daylight, "
    "wearing a casual jacket and jeans, fully clothed, relaxed pose",
    f"{training_run.trigger_word} is a woman taking a mirror selfie in a modern room, "
    "holding a phone, wearing a fully clothed everyday outfit",
    f"{training_run.trigger_word} is a woman seated and taking a mirror selfie, "
    "wearing a long-sleeve top and trousers, natural indoor lighting",
    f"{training_run.trigger_word} is a woman taking an indoor selfie near a window, "
    "wearing a knitted sweater, warm ambient light, clear facial features",
    f"{training_run.trigger_word} is a woman in a lifestyle photograph at a cafe, "
    "wearing a coat and scarf, fully clothed, candid expression",
]
SEEDS = [42, 12345, 987654321]
WIDTH = 1024
HEIGHT = 1024
INFERENCE_STEPS = 8
GUIDANCE_SCALE = 0.0
NEGATIVE_PROMPT = ""
CHECKPOINT_MODE = "auto"
MAXIMUM_CHECKPOINTS = 8
MANUAL_CHECKPOINT_STEPS = []
PRIMARY_ADAPTER_SCALE = 1.0
SCALE_SWEEP = [0.6, 0.8, 1.0]
COMPARE_BASE_MODEL = True
INCLUDE_BASE_IN_CHECKPOINT_GRID = True
RUN_CHECKPOINT_SWEEP = True
RUN_SCALE_SWEEP = True
INCLUDE_ALL_CHECKPOINTS_IN_EXPORT = False
DOWNLOAD_EXPORTS = False

evaluation_config = EvaluationConfig(
    prompts=PROMPTS,
    seeds=SEEDS,
    width=WIDTH,
    height=HEIGHT,
    inference_steps=INFERENCE_STEPS,
    guidance_scale=GUIDANCE_SCALE,
    negative_prompt=NEGATIVE_PROMPT,
    checkpoint_mode=CHECKPOINT_MODE,
    maximum_checkpoints=MAXIMUM_CHECKPOINTS,
    manual_checkpoint_steps=MANUAL_CHECKPOINT_STEPS,
    primary_adapter_scale=PRIMARY_ADAPTER_SCALE,
    scale_sweep=SCALE_SWEEP,
    compare_base_model=COMPARE_BASE_MODEL,
    include_base_in_checkpoint_grid=INCLUDE_BASE_IN_CHECKPOINT_GRID,
    run_checkpoint_sweep=RUN_CHECKPOINT_SWEEP,
    run_scale_sweep=RUN_SCALE_SWEEP,
)

pipeline.prepare_evaluation_assets()

evaluation = pipeline.evaluate(
    run=training_run,
    config=evaluation_config,
)

evaluation.show_base_comparison()
evaluation.show_checkpoint_grid()
evaluation.show_scale_grid()
evaluation.show_summary()

exports = evaluation.export(
    include_selected_lora=True,
    include_all_checkpoints=INCLUDE_ALL_CHECKPOINTS_IN_EXPORT,
    include_images=True,
    include_logs=True,
    include_manifests=True,
)

exports.display()

if DOWNLOAD_EXPORTS:
    exports.download()